# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip install -q -U "faiss-cpu>=1.8.0" chromadb sentence-transformers transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [2]:
DATA_PATH = Path("labelled_newscatcher_dataset.csv")
SUBSET_SIZE = 1000  # small enough to embed in seconds while iterating

pdf = pd.read_csv(DATA_PATH, sep=";")

# One stable integer id per article: FAISS's IndexIDMap needs int64 ids and Chroma
# needs string ids, so a single numeric column serves both (cast where needed).
if "id" not in pdf.columns:
    pdf["id"] = range(len(pdf))

display(pdf.head())

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


In [3]:
print(f"Shape: {pdf.shape}\n")
print(f"Dtypes:\n{pdf.dtypes}\n")
print(f"Missing values per column:\n{pdf.isna().sum()}\n")
print(f"Duplicate titles: {pdf['title'].duplicated().sum()}\n")
print(f"Articles per topic:\n{pdf['topic'].value_counts()}")

Shape: (108774, 7)

Dtypes:
topic             object
link              object
domain            object
published_date    object
title             object
lang              object
id                 int64
dtype: object

Missing values per column:
topic             0
link              0
domain            0
published_date    0
title             0
lang              0
id                0
dtype: int64

Duplicate titles: 5594

Articles per topic:
topic
TECHNOLOGY       15000
HEALTH           15000
WORLD            15000
ENTERTAINMENT    15000
BUSINESS         15000
SPORTS           15000
NATION           15000
SCIENCE           3774
Name: count, dtype: int64


In [4]:
# The first N rows (as the exercise asks) rather than a random sample, so results are
# reproducible. Trade-off: if the CSV is ordered by topic, the first 1000 rows may
# cover only a few topics, so we check the topic mix of the subset.
pdf_subset = pdf.head(SUBSET_SIZE).copy()

print(f"Topics in the subset:\n{pdf_subset['topic'].value_counts()}")
pdf_subset[["id", "title"]].head()

Topics in the subset:
topic
TECHNOLOGY    587
HEALTH        232
SCIENCE       181
Name: count, dtype: int64


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [5]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# One InputExample per title, using the article id as the guid so each example can be
# traced back to its row. Note: InputExample is the input format for *fine-tuning* a
# sentence-transformer; model.encode() in the next cell works on plain strings.
faiss_train_examples = [
    example_create_fn(idx, title)
    for idx, title in zip(pdf_subset["id"], pdf_subset["title"])
]
faiss_train_examples[:2]

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [7]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal

1000

In [8]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    """Return the k articles most similar to the query, best match first."""
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(query_vector, k)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

display(search_content('animal', pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [9]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

existing = [c if isinstance(c, str) else c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(
    name=collection_name, metadata={"hnsw:space": "cosine"}
)

N_DOCS = 100
docs = pdf_subset.head(N_DOCS)
collection.add(
    documents=docs["title"].tolist(),
    metadatas=[{"topic": topic} for topic in docs["topic"]],
    ids=docs["id"].astype(str).tolist(),
)

results = collection.query(query_texts=["space"], n_results=10)
print(json.dumps(results, indent=2))

{
  "ids": [
    [
      "72",
      "7",
      "30",
      "26",
      "23",
      "76",
      "69",
      "40",
      "47",
      "75"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
      "Orbital space tourism set for rebirth in 2021",
      "NASA drops \"insensitive\" nicknames for cosmic objects",
      "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
      "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
      "Australia's small yet crucial part in the mission to find life on Mars",
      "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
      "SpaceX's Starship spacecraft saw 150 meters high",
      "NASA\u2019s InSight lander shows what\u2019s beneath Mars\u2019 surface",
      "Alien base on Mercury: ET hunters claim to find hu

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [10]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"  # ~1 GB, runs on a free Colab CPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [11]:
SYSTEM_PROMPT = (
    "Answer the question using only the news headlines in the context. "
    "If the headlines don't contain the answer, say you don't know."
)

def answer_question(question: str, n_context: int = 3) -> tuple[list[str], str]:
    """Retrieve the n_context most relevant headlines from Chroma, then answer from them."""
    retrieved = collection.query(query_texts=[question], n_results=n_context)
    context_docs = retrieved["documents"][0]
    context = "\n".join(f"- {doc}" for doc in context_docs)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

    output = pipe(messages, do_sample=False)

    return context_docs, output[0]["generated_text"][-1]["content"]

question = "What's the latest news on space development?"
context_docs, response = answer_question(question)

print("CONTEXT:")
for doc in context_docs:
    print(f"  - {doc}")
print(f"\nANSWER: {response}")

CONTEXT:
  - Orbital space tourism set for rebirth in 2021
  - Beck teams up with NASA and AI for 'Hyperspace' visual album experience
  - SpaceX, NASA Demo-2 Rocket Launch Set for Saturday: How to Watch

ANSWER: Orbital space tourism is set for rebirth in 2021.


In [12]:
# The last question is probably not answerable from 100 headlines. It tests whether
# the model follows the "say you don't know" instruction or invents an answer.
experiments = [
    "What's the latest news on space development?",
    "What's new with animals and wildlife?",
    "Who won the last football World Cup?",
]

for question in experiments:
    for n_context in [1, 3, 10]:
        _, response = answer_question(question, n_context)
        print(f"Q: {question}  |  headlines in context: {n_context}")
        print(f"A: {response}\n")

Q: What's the latest news on space development?  |  headlines in context: 1
A: Orbital space tourism is set to re-birth in 2021.

Q: What's the latest news on space development?  |  headlines in context: 3
A: Orbital space tourism is set for rebirth in 2021.

Q: What's the latest news on space development?  |  headlines in context: 10
A: SpaceX Crew-1 mission with NASA, first fully operational crewed mission to space to launch in October

Q: What's new with animals and wildlife?  |  headlines in context: 1
A: New York City Zoo announces a new addition to its animal collection.

Q: What's new with animals and wildlife?  |  headlines in context: 3
A: Newsworthy Animals & Wildlife News Headlines:

Q: What's new with animals and wildlife?  |  headlines in context: 10
A: Tradeoff between the eyes and nose helps flies find their niche

Q: Who won the last football World Cup?  |  headlines in context: 1
A: The last football World Cup was won by Argentina.

Q: Who won the last football World C